In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
spark.sql("USE CATALOG olist_project")
spark.sql("USE SCHEMA bronze")



In [0]:
from pyspark.sql.types import StructType, StructField, StringType

order_schema=StructType([
    StructField('order_id', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('order_status', StringType(), True),
    StructField('order_purchase_timestamp', StringType(), True),
    StructField('order_approved_at', StringType(), True),
    StructField('order_delivered_carrier_date', StringType(), True),
    StructField('order_delivered_customer_date', StringType(), True),
    StructField('order_estimated_delivery_date', StringType(), True)
])

olist_orders_df = spark.read.schema(order_schema).option("header", True).option("mode", "FAILFAST").option("nullValue", "") .csv("/Volumes/olist_project/raw/olist_files/olist_orders_dataset.csv")
olist_orders_df.printSchema()
olist_orders_df.write.format("delta").mode("overwrite").saveAsTable("orders")


In [0]:
customer_schema=StructType([
    StructField('customer_id', StringType(), True),
    StructField('customer_unique_id', StringType(), True),
    StructField('customer_zip_code_prefix', StringType(), True),
    StructField('customer_city', StringType(), True),
    StructField('customer_state', StringType(), True)
])

customers_df = spark.read \
    .schema(customer_schema) \
    .option("header", True) \
    .option("mode", "FAILFAST") \
    .option("nullValue", "") \
    .csv("/Volumes/olist_project/raw/olist_files/olist_customers_dataset.csv")

customers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.bronze.customers")

In [0]:
order_items_schema=StructType([
    StructField('order_id', StringType(), True),
    StructField('order_item_id', StringType(), True),
    StructField('product_id', StringType(), True),
    StructField('seller_id', StringType(), True),
    StructField('shipping_limit_date', StringType(), True),
    StructField('price', StringType(), True),
    StructField('freight_value', StringType(), True)
])

order_items_df = spark.read \
    .schema(order_items_schema) \
    .option("header", True) \
    .option("mode", "FAILFAST") \
    .option("nullValue", "") \
    .csv("/Volumes/olist_project/raw/olist_files/olist_order_items_dataset.csv")

order_items_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.bronze.order_items")

In [0]:
order_payments_schema=StructType([
    StructField('order_id', StringType(), True),
    StructField('payment_sequential', StringType(), True),
    StructField('payment_type', StringType(), True),
    StructField('payment_installments', StringType(), True),
    StructField('payment_value', StringType(), True)
])

order_payments_df = spark.read \
    .schema(order_payments_schema) \
    .option("header", True) \
    .option("mode", "FAILFAST") \
    .option("nullValue", "") \
    .csv("/Volumes/olist_project/raw/olist_files/olist_order_payments_dataset.csv")

order_payments_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.bronze.payments")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

order_reviews_schema=StructType([
    StructField('review_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('review_score', StringType(), True),
    StructField('review_comment_title', StringType(), True),
    StructField('review_comment_message', StringType(), True),
    StructField('review_creation_date', StringType(), True),
    StructField('review_answer_timestamp', StringType(), True)
])

order_reviews_df = spark.read \
    .schema(order_reviews_schema) \
    .option("header", True) \
    .option("mode", "permissive") \
    .option("columnNameOfCorruptRecord", "_corrupt_record")\
    .csv("/Volumes/olist_project/raw/olist_files/olist_order_reviews_dataset.csv")
order_reviews_df.show()

In [0]:
order_reviews_schema=StructType([
    StructField('review_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('review_score', StringType(), True),
    StructField('review_comment_title', StringType(), True),
    StructField('review_comment_message', StringType(), True),
    StructField('review_creation_date', StringType(), True),
    StructField('review_answer_timestamp', StringType(), True)
])

order_reviews_df = spark.read \
    .schema(order_reviews_schema) \
    .option("header", True) \
    .option("mode", "FAILFAST") \
    .option("nullValue", "") \
    .option("multiLine", True)\
    .option("escape", "\"")\
    .csv("/Volumes/olist_project/raw/olist_files/olist_order_reviews_dataset.csv")

order_reviews_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.bronze.reviews")

In [0]:
product_schema=StructType([
    StructField('product_id', StringType(), True),
    StructField('product_category_name', StringType(), True),
    StructField('product_name_lenght', StringType(), True),
    StructField('product_description_lenght', StringType(), True),
    StructField('product_photos_qty', StringType(), True),
    StructField('product_weight_g', StringType(), True),
    StructField('product_length_cm', StringType(), True),
    StructField('product_height_cm', StringType(), True),
    StructField('product_width_cm', StringType(), True)
])
products_df = spark.read \
    .schema(product_schema) \
    .option("header", True) \
    .option("mode", "FAILFAST") \
    .option("nullValue", "") \
    .csv("/Volumes/olist_project/raw/olist_files/olist_products_dataset.csv")

products_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.bronze.products")

In [0]:
spark.sql("SELECT COUNT(*) FROM olist_project.bronze.orders_bronze").show()
spark.sql("SELECT COUNT(*) FROM olist_project.bronze.customers").show()
spark.sql("SELECT COUNT(*) FROM olist_project.bronze.order_items").show()
spark.sql("SELECT COUNT(*) FROM olist_project.bronze.order_payments").show()
spark.sql("SELECT COUNT(*) FROM olist_project.bronze.order_reviews").show()
spark.sql("SELECT COUNT(*) FROM olist_project.bronze.products").show()

In [0]:
%sql
DESCRIBE HISTORY olist_project.bronze.order_reviews;
